# 要約を ROUGE で評価する (日本語対応)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_evaluation/rouge_ja.ipynb)

NLP クックブック 20-03 ／ 評価指標と統計的検定

要約の出力を、参照要約との単語の重なり (ROUGE-1/2/L) で評価したい。日本語でも正しく計算したい。

In [1]:
!pip install -q rouge-score fugashi unidic-lite

In [2]:
from fugashi import Tagger
from rouge_score import rouge_scorer

class MecabTokenizer:
    """rouge-score の既定のトークナイザは英数字以外を捨ててしまうので、日本語用のものを渡す"""
    def __init__(self):
        self.tagger = Tagger("-Owakati")
    def tokenize(self, text):
        return self.tagger.parse(text).split()

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=MecabTokenizer())
ref = "政府は来年度予算案を閣議決定し、一般会計の総額は過去最大となった。"
for hyp in ["政府は来年度予算案を決定し、総額は過去最大となった。", "来年度の予算は過去最大の規模である。", "今日は天気が良い。"]:
    s = scorer.score(ref, hyp)    # 引数の順番は (参照, 出力)
    print(f"R1 {s['rouge1'].fmeasure:.3f}  R2 {s['rouge2'].fmeasure:.3f}  RL {s['rougeL'].fmeasure:.3f}  {hyp}")

R1 0.900  R2 0.789  RL 0.900  政府は来年度予算案を決定し、総額は過去最大となった。
R1 0.471  R2 0.187  RL 0.412  来年度の予算は過去最大の規模である。
R1 0.143  R2 0.000  RL 0.143  今日は天気が良い。


In [3]:
# 既定のトークナイザに日本語を渡すと、何も残らず 0 になる
default = rouge_scorer.RougeScorer(["rouge1"])
print(default.score(ref, "政府は来年度予算案を決定し、総額は過去最大となった。"))

# コーパス全体では、文書ごとの F 値を平均して報告するのが一般的
import numpy as np
refs = [ref, "新しい言語モデルが発表され、性能が大幅に向上した。"]
hyps = ["政府は予算案を決定した。", "新しいモデルが発表された。"]
print(round(float(np.mean([scorer.score(r, h)["rougeL"].fmeasure for r, h in zip(refs, hyps)])), 3))

{'rouge1': Score(precision=0.0, recall=0.0, fmeasure=0.0)}
0.624


- ROUGE-1/2 は単語 1-gram / 2-gram の重なり、ROUGE-L は最長共通部分列。再現率・適合率・F 値のどれを報告しているかを明記する。
- 日本語で「ROUGE が 0 になる」「異常に高い/低い」ときは、ほぼトークナイザの問題。